# 00 · Setup: download Qwen2.5-7B-Instruct to Google Drive

Downloads the large model into `models/` in your project folder (once). The small model, Qwen2.5-3B-Instruct, is already there.

- **Default:** `Qwen/Qwen2.5-7B-Instruct`, full precision, ~15 GB. Runs on an A100, G4 or L4.
- **Option:** the AWQ (compressed) version, ~5.5 GB, as used in the AgentTTS paper. If you use it, set `LARGE_MODEL_DIR = "Qwen2.5-7B-Instruct-AWQ"` in notebooks 04–06.
- **Option:** Qwen3-4B as the agent for Scripts 1 and 2 (thinking mode, ~8 GB). Not needed by default: the agent is Qwen2.5-7B.

Qwen models are not gated, so no licence approval is needed. A CPU runtime is fine.

In [ ]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
DOWNLOAD = {
    "Qwen2.5-7B-Instruct":     "Qwen/Qwen2.5-7B-Instruct",        # default large model (~15 GB)
    # "Qwen2.5-7B-Instruct-AWQ": "Qwen/Qwen2.5-7B-Instruct-AWQ",  # optional compressed version (~5.5 GB)
    # "Qwen3-4B":                "Qwen/Qwen3-4B",                  # optional agent with thinking mode (~8 GB)
}
DELETE_LLAMA = False   # True removes models/Llama-3.2-3B-Instruct (6.4 GB); the plan is Qwen-only now
ALLOW  = ["*.json", "*.safetensors", "tokenizer*", "*.model", "*.txt", "*.tiktoken"]
IGNORE = ["original/*", "*.pth", "*.bin", "*.gguf"]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, glob, time
!pip install -q -U huggingface_hub hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:                                    # optional: a token only raises download rate limits
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    tok = tok.get("value") if isinstance(tok, dict) else tok
except Exception:
    tok = None
from huggingface_hub import snapshot_download

MODEL_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)
def complete(p):
    return os.path.exists(f"{p}/config.json") and glob.glob(f"{p}/*.safetensors") and os.path.exists(f"{p}/tokenizer.json")

for name, repo in DOWNLOAD.items():
    dst = f"{MODEL_DIR}/{name}"
    if complete(dst):
        print(f"✓ {name} already in Drive"); continue
    t = time.time()
    tmp = f"/content/_dl/{name}"
    print(f"downloading {repo} ...")
    snapshot_download(repo_id=repo, local_dir=tmp, allow_patterns=ALLOW, ignore_patterns=IGNORE, token=tok or None)
    print(f"copying to {dst} ...")
    shutil.copytree(tmp, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
    shutil.rmtree(tmp)
    print(f"✓ {name} in {time.time() - t:.0f}s")

if DELETE_LLAMA and os.path.isdir(f"{MODEL_DIR}/Llama-3.2-3B-Instruct"):
    shutil.rmtree(f"{MODEL_DIR}/Llama-3.2-3B-Instruct"); print("deleted Llama-3.2-3B-Instruct")

In [ ]:
!du -sh "{MODEL_DIR}"/*
total, used, free = shutil.disk_usage("/content/drive")
print(f"Drive: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free")

Done. Next: **04_step0_results_table.ipynb** on an A100 GPU.